In [1]:
import pandas as pd
from time import perf_counter
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score

In [2]:
df = pd.read_csv('/kaggle/input/datasets/jatinkhandelwal112/indian-e-commerce-sales-analytics-dataset/sales.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Order_ID            250000 non-null  object 
 1   Customer_ID         250000 non-null  object 
 2   Product_ID          250000 non-null  object 
 3   Order_Date          250000 non-null  object 
 4   Order_Time          250000 non-null  object 
 5   Delivery_Date       250000 non-null  object 
 6   Quantity            250000 non-null  int64  
 7   Unit_Price          250000 non-null  float64
 8   Order_Value         250000 non-null  float64
 9   Shipping_Cost       250000 non-null  float64
 10  Coupon_Code         50185 non-null   object 
 11  Coupon_Discount     250000 non-null  float64
 12  Total_Amount        250000 non-null  float64
 13  Payment_Mode        250000 non-null  object 
 14  Order_Status        250000 non-null  object 
 15  Rating              120030 non-nul

In [3]:
df = df.fillna(df.mean(numeric_only=True))
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Order_ID            250000 non-null  object 
 1   Customer_ID         250000 non-null  object 
 2   Product_ID          250000 non-null  object 
 3   Order_Date          250000 non-null  object 
 4   Order_Time          250000 non-null  object 
 5   Delivery_Date       250000 non-null  object 
 6   Quantity            250000 non-null  int64  
 7   Unit_Price          250000 non-null  float64
 8   Order_Value         250000 non-null  float64
 9   Shipping_Cost       250000 non-null  float64
 10  Coupon_Code         50185 non-null   object 
 11  Coupon_Discount     250000 non-null  float64
 12  Total_Amount        250000 non-null  float64
 13  Payment_Mode        250000 non-null  object 
 14  Order_Status        250000 non-null  object 
 15  Rating              250000 non-nul

In [4]:
sizes = [1000, 10000, 50000, 100000]

for size in sizes:
    df_subset = df.sample(n=size, random_state=42)

    cols_to_use = df_subset.select_dtypes(include=['number']).columns.drop('Rating')
    X = df_subset[cols_to_use]
    y = df_subset['Rating']

    # Pipeline
    my_pipeline = Pipeline(steps=[
        ('preprocessor', SimpleImputer()),
        ('model', RandomForestRegressor(
            n_estimators=50,
            random_state=0
        ))
    ])

    # Duration
    t1 = perf_counter()

    scores = -1 * cross_val_score(
        my_pipeline,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )

    t2 = perf_counter()

    # Results
    print(f"\n===== {size:,} ROWS =====")
    print("MAE scores:", scores)
    print(f"Mean MAE: {scores.mean():.4f}")
    print(f"Duration: {t2-t1:.2f} second(s)")


===== 1,000 ROWS =====
MAE scores: [0.3654456  0.35883018 0.37400178 0.3732026  0.37201962]
Mean MAE: 0.3687
Duration: 1.34 second(s)

===== 10,000 ROWS =====
MAE scores: [0.37101581 0.38034245 0.37569791 0.38260447 0.38569073]
Mean MAE: 0.3791
Duration: 14.84 second(s)

===== 50,000 ROWS =====
MAE scores: [0.38532945 0.38679087 0.38595011 0.38545698 0.38629693]
Mean MAE: 0.3860
Duration: 79.55 second(s)

===== 100,000 ROWS =====
MAE scores: [0.38851692 0.38625821 0.38773218 0.38940928 0.38704067]
Mean MAE: 0.3878
Duration: 167.82 second(s)
